# slim-backtrader, worked through on AAPL and MSFT

This notebook walks through the whole package on two real stocks, Apple
(AAPL) and Microsoft (MSFT), with daily prices from Yahoo Finance through
[`yfinance`](https://github.com/ranaroussi/yfinance). It is built in the
order the package is: each section uses only what the sections before it
introduced.

| part | what it covers |
|---|---|
| 1. Data | downloading, caching, `PandasData`, `YahooFinanceCSVData`, date ranges, resampling, replay, filters |
| 2. Indicators | the built-in library, `next()` vs `once()`, writing your own |
| 3. A first strategy | the strategy life cycle, logging, `notify_order` / `notify_trade` |
| 4. Orders | market, close, limit, stop, stop-limit, trailing stops, validity, brackets, OCO, cancelling |
| 5. Sizing | sizers, and the `order_target_*` family |
| 6. The broker | commissions, slippage, cheat-on-close, interest on shorts |
| 7. Measuring | analyzers, observers, benchmarks, plotting |
| 8. Several stocks | rebalancing on a timer, momentum rotation, a market-neutral pair |
| 9. Signals and optimisation | `SignalStrategy`, `optstrategy`, a parameter heat map |
| 10. Option pricing | Black-Scholes, the binomial tree, implied volatility, the greeks |
| 11. Options on a model feed | covered calls, protective puts, cash-secured puts, spreads, collars |
| 12. An options book | combos, the cash-covered margin model, quote fills, portfolio greeks, P&L by greek |
| 13. Whole option chains | `OptionChain`, picking contracts by delta, stocks with an options overlay, delta hedging |
| 14. Summary | every strategy side by side |

**Requirements.** `pip install "slim-backtrader[plotting,pandas,options]" yfinance jupyter`.
Prices are cached in `data/` next to the notebook after the first download,
so later runs work offline and give the same numbers.

**A caveat that applies throughout.** Yahoo's free daily data has no option
prices, so every option in this notebook is *model-priced*: Black-Scholes or a
binomial tree on the stock's own realised volatility, with a quoted spread
around the model price. That is enough to study mechanics, risk and the shape
of a strategy's returns; it is not a substitute for historical option quotes.
There is no volatility skew, and no volatility risk premium, which flatters
option selling.

*Part of slim-backtrader, a modified version of backtrader (Copyright (C)
2015-2023 Daniel Rodriguez), Copyright (C) 2026 Dennis Hansen; free software
under the GNU General Public License, version 3 or later.*

In [ ]:
import datetime
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

import backtrader as bt
import backtrader.options as bo

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
pd.options.display.float_format = "{:,.4f}".format
print("backtrader", bt.__version__, "| yfinance", yf.__version__, "| pandas", pd.__version__)

The parameters of the whole notebook. Change the tickers or the dates here
and re-run everything; nothing below hard-codes either.

In [ ]:
TICKERS = ["AAPL", "MSFT"]
START, END = "2016-01-01", "2025-12-31"  # the stock sections
OPTIONS_START = "2023-01-01"  # the option sections use the last three years
DATA_DIR = Path("data")
CASH = 100_000.0
A, B = TICKERS

## 1. Data

### 1.1 Downloading and caching

`yf.download(..., auto_adjust=False)` returns the raw columns Yahoo
publishes, including `Adj Close`. The cache is written in Yahoo's own CSV
layout (`Date,Open,High,Low,Close,Adj Close,Volume`), which is exactly what
backtrader's `YahooFinanceCSVData` reads, so section 1.4 can load the same
file a second way and compare.

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
COLUMNS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]


def load_prices(ticker):
    """Daily prices for one ticker, from the cache or from Yahoo Finance."""
    path = DATA_DIR / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(
            ticker,
            start=START,
            end=END,
            auto_adjust=False,
            actions=False,
            progress=False,
            multi_level_index=False,
        )
        if raw.empty:
            raise RuntimeError(f"Yahoo Finance returned no data for {ticker}")
        raw.index.name = "Date"
        raw[COLUMNS].to_csv(path, float_format="%.6f")
    frame = pd.read_csv(path, index_col="Date", parse_dates=True)
    return frame.loc[START:END]


raw = {t: load_prices(t) for t in TICKERS}
for t, frame in raw.items():
    print(f"{t}: {len(frame)} days, {frame.index[0]:%Y-%m-%d} to {frame.index[-1]:%Y-%m-%d}")
raw[A].tail()

### 1.2 Adjusted prices

Yahoo's `Close` is adjusted for splits but not for dividends; `Adj Close`
is adjusted for both. A backtest that trades on `Close` books a loss every
time a dividend is paid, because the price drops and nothing pays it back.
The usual fix is to scale the whole bar by `Adj Close / Close`, which is
what `adjusted()` does. Volume is scaled the other way, so that
price × volume is unchanged.

In [ ]:
def adjusted(frame):
    factor = frame["Adj Close"] / frame["Close"]
    out = pd.DataFrame(index=frame.index)
    for col in ("Open", "High", "Low", "Close"):
        out[col.lower()] = frame[col] * factor
    out["volume"] = frame["Volume"] / factor
    out["openinterest"] = 0.0
    return out


prices = {t: adjusted(frame) for t, frame in raw.items()}

fig, ax = plt.subplots()
for t, frame in prices.items():
    ax.plot(frame.index, frame["close"] / frame["close"].iloc[0], label=t)
ax.set_yscale("log")
ax.set_title("Growth of 1 (adjusted close, log scale)")
ax.legend()
plt.show()

### 1.3 `PandasData`: a feed from a DataFrame

`PandasData` finds the columns by name (case-insensitive) and uses the
index as the bar's datetime. `fromdate` and `todate` cut the feed without
touching the frame. `feed()` is the helper the rest of the notebook uses.

Pass them as **dates**, not midnight datetimes. A daily bar is stamped at
the end of its session (23:59:59.999), so `todate=datetime(2025, 12, 31)`
means "up to midnight starting the 31st" and drops that day's bar, while
`todate=date(2025, 12, 31)` is completed to the session's end and keeps it.

In [ ]:
def feed(ticker, start=None, end=None, **kwargs):
    """A PandasData feed of a ticker's adjusted bars, named after the ticker."""
    kw = dict(dataname=prices[ticker], name=ticker)
    if start:
        kw["fromdate"] = pd.Timestamp(start).date()
    if end:
        kw["todate"] = pd.Timestamp(end).date()
    kw.update(kwargs)
    return bt.feeds.PandasData(**kw)


class Bars(bt.Strategy):
    """Records every bar of every feed it is given."""

    def __init__(self):
        self.rows = {d._name: [] for d in self.datas}
        self.seen = {d._name: 0 for d in self.datas}

    def prenext(self):
        self.next()

    def next(self):
        for d in self.datas:
            if len(d) > self.seen[d._name]:  # a new bar on this feed
                self.seen[d._name] = len(d)
                self.rows[d._name].append(
                    (d.datetime.date(0), d.open[0], d.high[0], d.low[0], d.close[0], d.volume[0])
                )

    def frames(self):
        cols = ["date", "open", "high", "low", "close", "volume"]
        return {
            k: pd.DataFrame(v, columns=cols).set_index("date") for k, v in self.rows.items()
        }


def bars_of(*datas, **cerebro_kwargs):
    cerebro = bt.Cerebro(stdstats=False, **cerebro_kwargs)
    for d in datas:
        cerebro.adddata(d)
    cerebro.addstrategy(Bars)
    return cerebro.run()[0].frames()


year = str(pd.Timestamp(END).year - 1)
seen = bars_of(feed(A, start=f"{year}-01-01", end=f"{year}-12-31"))[A]
print(f"{A} in {year}: {len(seen)} bars, first {seen.index[0]}, last {seen.index[-1]}")
seen.head(3)

### 1.4 The same file through `YahooFinanceCSVData`

`YahooFinanceCSVData` reads Yahoo's CSV layout directly. With
`adjclose=True` (the default) it applies the same `Adj Close / Close`
adjustment as `adjusted()` above, and `round=True` (also the default) rounds
prices to 2 decimals. The two feeds should agree to that rounding.

In [ ]:
csv_feed = bt.feeds.YahooFinanceCSVData(
    dataname=str(DATA_DIR / f"{A}.csv"),
    fromdate=pd.Timestamp(START).date(),
    todate=pd.Timestamp(END).date(),
    name=f"{A}-csv",
)
both = bars_of(feed(A), csv_feed)
gap = (both[A] - both[f"{A}-csv"]).abs()
print("bars:", len(both[A]), "vs", len(both[f"{A}-csv"]))
print("largest price difference:", gap[["open", "high", "low", "close"]].max().max())
print("largest relative volume difference:", (gap["volume"] / both[A]["volume"]).max())

### 1.5 Resampling and replay

`resampledata` turns daily bars into weekly or monthly ones; the strategy
sees one bar per completed period. `replaydata` delivers the same weekly
bar but *updated every day* while the week is forming, which is how a
strategy on a higher timeframe sees an unfinished bar in live trading.

In [ ]:
cerebro = bt.Cerebro(stdstats=False)
daily = feed(A)
cerebro.adddata(daily)
cerebro.resampledata(daily, timeframe=bt.TimeFrame.Weeks, name=f"{A}-weekly")
cerebro.resampledata(daily, timeframe=bt.TimeFrame.Months, name=f"{A}-monthly")
cerebro.addstrategy(Bars)
frames = cerebro.run()[0].frames()
for name, frame in frames.items():
    print(f"{name:12s} {len(frame):5d} bars")

check = prices[A].resample("ME").agg(
    {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
)
mine = frames[f"{A}-monthly"]
print(
    "monthly bars agree with pandas' own resample:",
    np.allclose(mine[["open", "high", "low", "close"]].values, check[["open", "high", "low", "close"]].values[: len(mine)]),
)

In [ ]:
class WatchReplay(bt.Strategy):
    def __init__(self):
        self.seen = []

    def next(self):
        self.seen.append((self.data.datetime.date(0), len(self.data), self.data.close[0]))


cerebro = bt.Cerebro(stdstats=False)
cerebro.replaydata(feed(A, start=f"{year}-01-01", end=f"{year}-02-28"), timeframe=bt.TimeFrame.Weeks)
cerebro.addstrategy(WatchReplay)
replayed = pd.DataFrame(cerebro.run()[0].seen, columns=["day", "weekly bar #", "close so far"])
replayed.head(8)

### 1.6 Filters

A filter rewrites a feed's bars before any strategy sees them.
`HeikinAshi` smooths candles; `Renko` keeps only moves of a fixed size;
`DataFilter` wraps a feed and keeps the bars a function accepts.

In [ ]:
ha = feed(A, start=f"{year}-01-01", end=f"{year}-12-31", name=f"{A}-heikin-ashi")
ha.addfilter(bt.filters.HeikinAshi)
renko = feed(A, start=f"{year}-01-01", end=f"{year}-12-31", name=f"{A}-renko")
size = round(float(prices[A]["close"].loc[year].mean()) * 0.02, 2)
renko.addfilter(bt.filters.Renko, size=size)
mondays = bt.filters.DataFilter(
    dataname=feed(A, start=f"{year}-01-01", end=f"{year}-12-31"),
    funcfilter=lambda d: d.datetime.date(0).weekday() == 0,
    name=f"{A}-mondays",
)
out = bars_of(ha, renko, mondays, preload=False)
for name, frame in out.items():
    print(f"{name:18s} {len(frame):4d} bars")
print(f"renko brick size: {size}")
out[f"{A}-heikin-ashi"].head(3)

### 1.7 Reading a feed inside a strategy

Lines are indexed *relative to now*: `close[0]` is today's close,
`close[-1]` yesterday's. `get(size=n)` returns the last `n` values, `len()`
the number of bars seen so far, and `datetime.date(0)` today's date.

In [ ]:
class Peek(bt.Strategy):
    def next(self):
        if len(self.data) == 5:
            d = self.data
            print("date        ", d.datetime.date(0))
            print("close[0]    ", round(d.close[0], 2), " close[-1]", round(d.close[-1], 2))
            print("last 5 close", [round(x, 2) for x in d.close.get(size=5)])
            print("bars seen   ", len(d), "of", d.buflen())
            print("line names  ", d.lines.getlinealiases())


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(feed(A, start=f"{year}-01-01"))
cerebro.addstrategy(Peek)
_ = cerebro.run()

## 2. Indicators

### 2.1 The built-in library

There are over 300 indicator classes (counting aliases). An indicator is
declared in a strategy's `__init__` and computed by the engine; the strategy
reads its current value as `ind[0]`, exactly like a data line.

`Record` below is a small utility used throughout: it takes a function that
declares indicators on a feed and returns `{name: line}`, and records every
line on every bar into a DataFrame.

In [ ]:
print(len([n for n in dir(bt.indicators) if not n.startswith("_")]), "names in bt.indicators, e.g.")
print(", ".join(sorted(n for n in dir(bt.indicators) if n[:1].isupper())[:40]), "...")


class Record(bt.Strategy):
    params = dict(make=None)

    def __init__(self):
        self.lines_to_record = self.p.make(self.data)
        self.rows = []

    def next(self):
        row = {"date": self.data.datetime.date(0)}
        for name, line in self.lines_to_record.items():
            row[name] = line[0]
        self.rows.append(row)

    def frame(self):
        return pd.DataFrame(self.rows).set_index("date")


def record(make, data, **cerebro_kwargs):
    cerebro = bt.Cerebro(stdstats=False, **cerebro_kwargs)
    cerebro.adddata(data)
    cerebro.addstrategy(Record, make=make)
    return cerebro.run()[0].frame()


def panel(d):
    bb = bt.ind.BollingerBands(d, period=20, devfactor=2.0)
    macd = bt.ind.MACD(d, period_me1=12, period_me2=26, period_signal=9)
    stoch = bt.ind.Stochastic(d, period=14)
    return {
        "close": d.close,
        "sma50": bt.ind.SMA(d, period=50),
        "sma200": bt.ind.SMA(d, period=200),
        "ema20": bt.ind.EMA(d, period=20),
        "bb_top": bb.top,
        "bb_bot": bb.bot,
        "rsi": bt.ind.RSI(d, period=14),
        "macd": macd.macd,
        "signal": macd.signal,
        "atr": bt.ind.ATR(d, period=14),
        "stoch_k": stoch.percK,
        "stoch_d": stoch.percD,
        "cross": bt.ind.CrossOver(bt.ind.SMA(d, period=50), bt.ind.SMA(d, period=200)),
    }


ind = record(panel, feed(A))
print("first bar with every indicator valid:", ind.index[0], "(the 200-day SMA needs 200 bars)")
ind.tail()

In [ ]:
last = ind.iloc[-504:]  # the last two years
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 9), sharex=True, height_ratios=[3, 1, 1])
ax1.plot(last.index, last.close, label="close", color="black", lw=1)
ax1.plot(last.index, last.sma50, label="SMA 50")
ax1.plot(last.index, last.sma200, label="SMA 200")
ax1.fill_between(last.index, last.bb_bot, last.bb_top, alpha=0.15, label="Bollinger (20, 2)")
for day in last.index[last.cross != 0]:
    ax1.axvline(day, color="green" if last.cross[day] > 0 else "red", ls="--", lw=1)
ax1.set_title(f"{A}: moving averages, Bollinger bands and 50/200 crossings")
ax1.legend(loc="upper left")
ax2.plot(last.index, last.rsi, color="purple")
ax2.axhline(70, color="grey", ls=":")
ax2.axhline(30, color="grey", ls=":")
ax2.set_ylabel("RSI 14")
ax3.plot(last.index, last.macd, label="MACD")
ax3.plot(last.index, last.signal, label="signal")
ax3.bar(last.index, last.macd - last.signal, color="grey", alpha=0.5)
ax3.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 2.2 Checking the arithmetic

An indicator library is only useful if its numbers are right. The simple
moving average must equal pandas' rolling mean; the exponential average
seeds with a simple average and then follows `a·x + (1-a)·prev`, which
pandas reproduces with `ewm(adjust=False)` once both start from the same
seed. The true range is `max(high, prev close) - min(low, prev close)`.

In [ ]:
full = record(lambda d: {"sma50": bt.ind.SMA(d, period=50), "ema20": bt.ind.EMA(d, period=20)}, feed(A))
close = prices[A]["close"]
sma_pd = close.rolling(50).mean()
seed = close.iloc[:20].mean()
ema_pd = pd.concat([pd.Series([seed], index=[close.index[19]]), close.iloc[20:]]).ewm(alpha=2 / 21, adjust=False).mean()
full.index = pd.to_datetime(full.index)
print("SMA 50 vs pandas rolling mean, max abs diff:", float((full.sma50 - sma_pd.reindex(full.index)).abs().max()))
print("EMA 20 vs pandas ewm,          max abs diff:", float((full.ema20 - ema_pd.reindex(full.index)).abs().max()))

### 2.3 Two execution paths, one answer

Every indicator can be computed bar by bar (`next()`) or vectorised over the
whole preloaded array (`once()`, used when `runonce=True`, the default).
`exactbars` trades memory for history: `exactbars=1` keeps only as many bars
as the longest indicator needs. All of these must give the same numbers.

In [ ]:
modes = {
    "runonce": dict(),
    "next, preloaded": dict(runonce=False),
    "next, streamed": dict(runonce=False, preload=False),
    "exactbars=1": dict(exactbars=1),
}
results = {name: record(panel, feed(A), **kw) for name, kw in modes.items()}
base = results["runonce"]
for name, frame in results.items():
    same_shape = frame.shape == base.shape
    diff = float((frame - base).abs().max().max()) if same_shape else float("nan")
    print(f"{name:16s} rows {len(frame):5d}   largest difference from runonce: {diff:.2e}")

### 2.4 Writing your own

An indicator declares its `lines` and `params`. The simplest way to write
one is *declaratively*, in `__init__`, from other lines: the framework then
derives both `next()` and `once()` and the minimum period. When a
calculation does not decompose like that, write `next()` yourself and set
the minimum period with `addminperiod`.

Below: a rolling z-score written both ways, and Kaufman's efficiency ratio
(net move over the sum of absolute moves), which needs `next()`.

In [ ]:
class ZScore(bt.Indicator):
    lines = ("z",)
    params = dict(period=20)
    plotinfo = dict(plothlines=[-2, 0, 2])

    def __init__(self):
        mean = bt.ind.SMA(self.data, period=self.p.period)
        std = bt.ind.StdDev(self.data, period=self.p.period)
        self.lines.z = (self.data - mean) / std


class ZScoreByHand(bt.Indicator):
    lines = ("z",)
    params = dict(period=20)

    def __init__(self):
        self.addminperiod(self.p.period)

    def next(self):
        window = np.array(self.data.get(size=self.p.period))
        self.lines.z[0] = (window[-1] - window.mean()) / window.std()


class EfficiencyRatio(bt.Indicator):
    lines = ("er",)
    params = dict(period=10)

    def __init__(self):
        self.addminperiod(self.p.period + 1)

    def next(self):
        window = np.array(self.data.get(size=self.p.period + 1))
        path = np.abs(np.diff(window)).sum()
        self.lines.er[0] = abs(window[-1] - window[0]) / path if path else 0.0


mine = record(
    lambda d: {
        "z": ZScore(d.close, period=20),
        "z_by_hand": ZScoreByHand(d.close, period=20),
        "er": EfficiencyRatio(d.close, period=10),
    },
    feed(A),
)
print("declarative vs next(), max abs diff:", float((mine.z - mine.z_by_hand).abs().max()))
print("first valid bar:", mine.index[0], "(20 bars in)")
mine.describe().loc[["mean", "std", "min", "max"]]

## 3. A first strategy

A strategy is a class with up to four hooks: `__init__` declares
indicators, `next()` runs once per bar and trades, `notify_order()` hears
about every order's progress, and `notify_trade()` about every round trip.
`prenext()` runs instead of `next()` while the indicators are still warming
up, and `stop()` once at the end.

The classic 50/200-day moving-average crossover on AAPL: long when the fast
average crosses above the slow one, flat (or short, with `short=True`) when
it crosses back.

In [ ]:
class SmaCross(bt.Strategy):
    params = dict(fast=50, slow=200, short=False, target=0.95, printlog=False)

    def __init__(self):
        fast = bt.ind.SMA(self.data, period=self.p.fast)
        slow = bt.ind.SMA(self.data, period=self.p.slow)
        self.cross = bt.ind.CrossOver(fast, slow)
        self.log_lines = []

    def log(self, text):
        line = f"{self.data.datetime.date(0)} {text}"
        self.log_lines.append(line)
        if self.p.printlog:
            print(line)

    def next(self):
        if self.cross > 0:
            self.order_target_percent(target=self.p.target)
        elif self.cross < 0:
            self.order_target_percent(target=-self.p.target if self.p.short else 0.0)

    def notify_order(self, order):
        if order.status in (order.Submitted, order.Accepted):
            return  # nothing to report yet
        if order.status == order.Completed:
            side = "BUY " if order.isbuy() else "SELL"
            # executed.value is the cost of what the order opened or closed;
            # size x price is what changed hands
            amount = abs(order.executed.size) * order.executed.price
            self.log(
                f"{side} {order.executed.size:+8.0f} @ {order.executed.price:9.2f} "
                f"= {amount:12,.2f}, commission {order.executed.comm:7.2f}"
            )
        else:
            self.log(f"order {order.ref} {order.getstatusname()}")

    def notify_trade(self, trade):
        if trade.isclosed:
            self.log(f"TRADE closed: gross {trade.pnl:+12,.2f}, net {trade.pnlcomm:+12,.2f}")

    def stop(self):
        self.log(f"final value {self.broker.getvalue():,.2f}")


cerebro = bt.Cerebro()
cerebro.adddata(feed(A))
cerebro.addstrategy(SmaCross)
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)  # 0.1% of the traded value
strat = cerebro.run()[0]
print("\n".join(strat.log_lines[:8]), "\n...\n" + "\n".join(strat.log_lines[-3:]))

buy_hold = CASH * prices[A]["close"].iloc[-1] / prices[A]["close"].iloc[0]
print(f"\nSMA cross: {cerebro.broker.getvalue():,.0f}   buy and hold: {buy_hold:,.0f}")

`cerebro.plot()` draws the run: the price with the strategy's indicators,
the buy and sell markers, the account value and the closed trades. It shows
the figure inline here; `start` and `end` take dates to zoom in.

In [ ]:
def show(cerebro, **kwargs):
    """cerebro.plot() at a size that suits a notebook."""
    with plt.rc_context({"figure.figsize": (14, 8), "figure.dpi": 90}):
        cerebro.plot(iplot=True, **kwargs)


show(cerebro, style="candle", barup="green", bardown="red", volume=False)

## 4. Orders

### 4.1 A tour of the order types

Each scenario below places one kind of order on a fixed bar and records
what the broker did with it. Reading the table:

- **Market** fills at the *next* bar's open: the strategy decides on the
  close of bar *t* and cannot trade at a price it has already seen.
- **Close** fills at the next bar's close.
- **Limit** buys at the limit or better: at the open if the bar gaps below
  it, else at the limit itself when the bar trades through it.
- **Stop** buys once the price rises to the stop (a breakout entry).
- **StopLimit** turns into a limit order once the stop is touched.
- **StopTrail** follows the price at a fixed distance and fires on the
  pull-back.
- `valid=` makes an order expire; `cancel()` withdraws it.

In [ ]:
class OrderTour(bt.Strategy):
    params = dict(scenario="market", day=30)

    def __init__(self):
        self.events = []
        self.entry = None

    def next(self):
        if len(self.data) != self.p.day:
            return
        c = self.data.close[0]
        s = self.p.scenario
        if s == "market":
            self.buy(size=100)
        elif s == "close":
            self.buy(size=100, exectype=bt.Order.Close)
        elif s == "limit 3% below":
            self.buy(size=100, exectype=bt.Order.Limit, price=c * 0.97, valid=datetime.timedelta(days=30))
        elif s == "limit 20% below, 5 days":
            self.buy(size=100, exectype=bt.Order.Limit, price=c * 0.80, valid=datetime.timedelta(days=5))
        elif s == "stop 2% above":
            self.buy(size=100, exectype=bt.Order.Stop, price=c * 1.02)
        elif s == "stoplimit":
            self.buy(size=100, exectype=bt.Order.StopLimit, price=c * 1.02, plimit=c * 1.03)
        elif s == "trailing stop 5%":
            self.buy(size=100)
            self.sell(size=100, exectype=bt.Order.StopTrail, trailpercent=0.05)
        elif s == "cancelled":
            order = self.buy(size=100, exectype=bt.Order.Limit, price=c * 0.5)
            self.cancel(order)

    def notify_order(self, order):
        self.events.append(
            dict(
                scenario=self.p.scenario,
                ref=order.ref,
                type=order.getordername(),
                side="buy" if order.isbuy() else "sell",
                status=order.getstatusname(),
                date=self.data.datetime.date(0),
                created_price=order.created.price,
                executed_price=order.executed.price if order.status == order.Completed else None,
            )
        )


scenarios = [
    "market", "close", "limit 3% below", "limit 20% below, 5 days",
    "stop 2% above", "stoplimit", "trailing stop 5%", "cancelled",
]
rows = []
for s in scenarios:
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(feed(A, start=f"{year}-01-01", end=f"{year}-12-31"))
    cerebro.addstrategy(OrderTour, scenario=s)
    cerebro.broker.setcash(CASH)
    rows += cerebro.run()[0].events

tour = pd.DataFrame(rows)
decision = prices[A].loc[year].iloc[29]
print(f"decision bar: {decision.name:%Y-%m-%d}, close {decision['close']:.2f}")
tour[~tour.status.isin(["Submitted", "Accepted"])].set_index("scenario")

### 4.2 Brackets and OCO

`buy_bracket()` sends three linked orders: the entry, a take-profit limit
above and a protective stop below. The two exits are one-cancels-other:
whichever fills first cancels its sibling, and neither is active until the
entry has filled. `oco=` links any two orders the same way.

In [ ]:
class Brackets(bt.Strategy):
    params = dict(every=20, profit=0.06, loss=0.03)

    def __init__(self):
        self.outcomes = []
        self.legs = {}

    def next(self):
        if self.position or len(self.data) % self.p.every:
            return
        c = self.data.close[0]
        entry, stop, limit = self.buy_bracket(
            size=100, price=c, exectype=bt.Order.Limit,
            stopprice=c * (1 - self.p.loss), limitprice=c * (1 + self.p.profit),
        )
        self.legs[stop.ref] = "stop-loss"
        self.legs[limit.ref] = "take-profit"

    def notify_order(self, order):
        if order.status == order.Completed and order.ref in self.legs:
            self.outcomes.append(self.legs[order.ref])


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(feed(A))
cerebro.addstrategy(Brackets)
cerebro.broker.setcash(CASH)
st = cerebro.run()[0]
counts = pd.Series(st.outcomes).value_counts()
print(f"{len(st.outcomes)} brackets closed:", dict(counts))
print(f"final value {cerebro.broker.getvalue():,.0f}")

In [ ]:
class Breakout(bt.Strategy):
    """A buy stop above the range and a sell stop below it, linked by OCO."""

    params = dict(lookback=20)

    def __init__(self):
        self.hi = bt.ind.Highest(self.data.high, period=self.p.lookback)
        self.lo = bt.ind.Lowest(self.data.low, period=self.p.lookback)
        self.pending = []
        self.fills = []

    def next(self):
        if self.position:
            if len(self) - self.opened >= 10:
                self.close()
            return
        for o in self.pending:
            self.cancel(o)
        up = self.buy(size=100, exectype=bt.Order.Stop, price=self.hi[0])
        down = self.sell(size=100, exectype=bt.Order.Stop, price=self.lo[0], oco=up)
        self.pending = [up, down]

    def notify_order(self, order):
        if order.status == order.Completed and order in self.pending:
            self.opened = len(self)
            self.fills.append("long" if order.isbuy() else "short")
        if order.status == order.Cancelled and order in self.pending and self.position:
            pass  # the sibling went when the other side filled


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(feed(B))
cerebro.addstrategy(Breakout)
cerebro.broker.setcash(CASH)
st = cerebro.run()[0]
print(f"{B} 20-day breakouts: {pd.Series(st.fills).value_counts().to_dict()}, final value {cerebro.broker.getvalue():,.0f}")

## 5. Sizing

A `buy()` without a size asks the strategy's **sizer**. `FixedSize` always
trades the same number of shares, `PercentSizer` a percentage of the cash,
`AllInSizer` all of it (the `Int` variants round down to whole shares). The
`order_target_*` methods are the other approach: state the position you
want - in shares, in money or as a fraction of the account - and the
strategy sends whatever order gets there.

In [ ]:
class SizedCross(bt.Strategy):
    params = dict(fast=50, slow=200)

    def __init__(self):
        self.cross = bt.ind.CrossOver(bt.ind.SMA(period=self.p.fast), bt.ind.SMA(period=self.p.slow))
        self.sizes = []

    def next(self):
        if self.cross > 0 and not self.position:
            self.buy()
        elif self.cross < 0 and self.position:
            self.close()

    def notify_order(self, order):
        if order.status == order.Completed and order.isbuy():
            self.sizes.append(order.executed.size)


rows = []
for label, sizer, kw in [
    ("FixedSize(100)", bt.sizers.FixedSize, dict(stake=100)),
    ("PercentSizer(50%)", bt.sizers.PercentSizer, dict(percents=50)),
    ("PercentSizerInt(95%)", bt.sizers.PercentSizerInt, dict(percents=95)),
    ("AllInSizerInt", bt.sizers.AllInSizerInt, dict()),
]:
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(feed(A))
    cerebro.addstrategy(SizedCross)
    cerebro.addsizer(sizer, **kw)
    cerebro.broker.setcash(CASH)
    st = cerebro.run()[0]
    rows.append(dict(sizer=label, first_buy=st.sizes[0] if st.sizes else None,
                     buys=len(st.sizes), final_value=cerebro.broker.getvalue()))
pd.DataFrame(rows).set_index("sizer")

In [ ]:
class Targets(bt.Strategy):
    """Walks a position through order_target_size, _value and _percent."""

    def __init__(self):
        self.steps = []

    def next(self):
        n = len(self)
        if n == 10:
            self.order_target_size(target=50)
        elif n == 20:
            self.order_target_value(target=20_000)
        elif n == 30:
            self.order_target_percent(target=0.50)
        elif n == 40:
            self.order_target_percent(target=0.0)

    def notify_order(self, order):
        if order.status == order.Completed:
            self.steps.append(dict(
                bar=len(self), size=order.executed.size, price=order.executed.price,
                position_after=self.position.size,
                position_value=self.position.size * self.data.close[0],
                account=self.broker.getvalue(),
            ))


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(feed(A, start=f"{year}-01-01"))
cerebro.addstrategy(Targets)
cerebro.broker.setcash(CASH)
pd.DataFrame(cerebro.run()[0].steps)

## 6. The broker

The default broker fills every order exactly at the price it should and
charges nothing. Real trading is less kind. The same long/short crossover
runs below under increasingly realistic assumptions:

- a **percentage commission** (0.1% of the traded value);
- a **per-share commission** with a minimum, written as a `CommInfoBase`
  subclass, the way US brokers charge;
- **slippage**: each fill 0.05% worse than the bar's price;
- **cheat-on-close**: market orders fill at the close of the bar that
  decided them (`set_coc`) - convenient, and optimistic;
- **interest on short positions**: 3% a year on the value of the short.

In [ ]:
class PerShare(bt.CommInfoBase):
    params = dict(per_share=0.005, minimum=1.0, stocklike=True, commtype=bt.CommInfoBase.COMM_FIXED)

    def _getcommission(self, size, price, pseudoexec):
        return max(self.p.minimum, abs(size) * self.p.per_share)


def broker_run(setup):
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(feed(A))
    cerebro.addstrategy(SmaCross, short=True)
    cerebro.broker.setcash(CASH)
    setup(cerebro.broker)
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name="trades")
    st = cerebro.run()[0]
    comm = sum(o.executed.comm for o in st.broker.orders if o.status == o.Completed)
    return dict(final_value=cerebro.broker.getvalue(), commissions=comm,
                trades=st.analyzers.trades.get_analysis().total.closed)


setups = {
    "frictionless": lambda b: None,
    "0.1% commission": lambda b: b.setcommission(commission=0.001),
    "$0.005/share, $1 minimum": lambda b: b.addcommissioninfo(PerShare()),
    "0.1% + 0.05% slippage": lambda b: (b.setcommission(commission=0.001), b.set_slippage_perc(0.0005)),
    "cheat-on-close": lambda b: (b.setcommission(commission=0.001), b.set_coc(True)),
    "0.1% + 3% short interest": lambda b: b.setcommission(commission=0.001, interest=0.03),
}
pd.DataFrame({name: broker_run(s) for name, s in setups.items()}).T

### 6.1 Cheat-on-open

A strategy that decides on the close normally trades at the next open, at
a price it has not seen, so it sizes on the close and the overnight gap
decides what it actually spends. With `cheat_on_open=True` cerebro calls
`next_open()` once the new bar's open is known and before the broker
fills, so the order can be sized to that open - here to spend exactly 95%
of the account.

One subtlety: inside `next_open()` the *data* already show the new bar,
but the *indicators* have not moved yet, so `cross[0]` is still the
crossover of the bar that just closed. The fills land on the same days as
before; only the sizing changes.

In [ ]:
class CrossOnOpen(SmaCross):
    def next(self):
        pass  # decisions move to next_open

    def next_open(self):
        if self.cross[0] > 0:  # the bar that just closed crossed up
            self.buy(size=int(0.95 * self.broker.getvalue() / self.data.open[0]))
        elif self.cross[0] < 0 and self.position:
            self.close()


rows = {}
for label, cls, kw in [("next() -> next open", SmaCross, {}), ("next_open() -> this open", CrossOnOpen, dict(cheat_on_open=True))]:
    cerebro = bt.Cerebro(stdstats=False, **kw)
    cerebro.adddata(feed(A))
    cerebro.addstrategy(cls)
    cerebro.broker.setcash(CASH)
    cerebro.broker.setcommission(commission=0.001)
    st = cerebro.run()[0]
    buys = [o for o in st.broker.orders if o.status == o.Completed and o.isbuy()]
    rows[label] = dict(first_buy=buys[0].executed.dt and bt.num2date(buys[0].executed.dt).date(),
                       price=buys[0].executed.price, invested=buys[0].executed.value,
                       invested_pct=100 * buys[0].executed.value / CASH,
                       final_value=cerebro.broker.getvalue())
pd.DataFrame(rows).T

### 6.2 Fund mode: deposits that are not returns

Adding cash to an account raises its value without any investment gain.
Measured on the account value, a deposit looks like a profit. In **fund
mode** the broker keeps a share count, like a mutual fund: money in or out
buys or redeems shares at the current share value, and returns are
measured on the share value. Below, 50,000 is added halfway through a
buy-and-hold run and invested the next day (`add_cash` reaches the account
on the broker's next cycle).

In [ ]:
class HoldAndDeposit(bt.Strategy):
    def next(self):
        if len(self) == 1:
            self.order_target_percent(target=0.95)
        half = len(self.data.array) // 2
        if len(self) == half:
            self.broker.add_cash(50_000)
        elif len(self) == half + 1:
            self.order_target_percent(target=0.95)  # invest the deposit


rows = {}
for fund in (False, True):
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(feed(A))
    cerebro.addstrategy(HoldAndDeposit)
    cerebro.broker.setcash(CASH)
    cerebro.broker.set_fundmode(fund, fundstartval=100.0)
    cerebro.addanalyzer(bt.analyzers.TimeReturn, _name="total", timeframe=bt.TimeFrame.NoTimeFrame)
    st = cerebro.run()[0]
    rows["fund mode" if fund else "account value"] = dict(
        final_value=cerebro.broker.getvalue(),
        measured_return_pct=100 * list(st.analyzers.total.get_analysis().values())[0],
        fund_shares=cerebro.broker.get_fundshares() if fund else None,
        share_value=cerebro.broker.get_fundvalue() if fund else None,
    )
print(f"{A} itself returned {100 * (prices[A]['close'].iloc[-1] / prices[A]['close'].iloc[0] - 1):.1f}% over the period")
pd.DataFrame(rows).T

## 7. Measuring: analyzers, observers and plots

### 7.1 Analyzers

An **analyzer** watches a run and reports at the end through
`get_analysis()`. `add_analyzers()` attaches the standard set, and
`summary()` turns a finished strategy into one row of numbers. Every
strategy from here on is reported through these two functions, so the
results can be compared in section 14.

In [ ]:
def add_analyzers(cerebro):
    an = bt.analyzers
    cerebro.addanalyzer(an.Returns, _name="returns", tann=252)
    cerebro.addanalyzer(an.SharpeRatio_A, _name="sharpe", timeframe=bt.TimeFrame.Days, riskfreerate=0.0)
    cerebro.addanalyzer(an.DrawDown, _name="drawdown")
    cerebro.addanalyzer(an.TimeDrawDown, _name="timedd", timeframe=bt.TimeFrame.Days)
    cerebro.addanalyzer(an.TradeAnalyzer, _name="trades")
    cerebro.addanalyzer(an.SQN, _name="sqn")
    cerebro.addanalyzer(an.Calmar, _name="calmar")
    cerebro.addanalyzer(an.VWR, _name="vwr")
    cerebro.addanalyzer(an.AnnualReturn, _name="annual")
    cerebro.addanalyzer(an.TimeReturn, _name="daily", timeframe=bt.TimeFrame.Days)
    cerebro.addanalyzer(an.TimeReturn, _name="monthly", timeframe=bt.TimeFrame.Months)
    cerebro.addanalyzer(an.PeriodStats, _name="periods", timeframe=bt.TimeFrame.Years)
    cerebro.addanalyzer(an.GrossLeverage, _name="leverage")
    cerebro.addanalyzer(an.PositionsValue, _name="positions")
    cerebro.addanalyzer(an.Transactions, _name="transactions")


def equity(strat, start_value=None):
    """The account value through the run, from the daily TimeReturn analyzer."""
    r = pd.Series(strat.analyzers.daily.get_analysis())
    r.index = pd.to_datetime(r.index)
    return (start_value or strat.broker.startingcash) * (1 + r).cumprod()


def summary(strat, name):
    a = strat.analyzers
    trades = a.trades.get_analysis()
    closed = trades.get("total", {}).get("closed", 0)
    won = trades.get("won", {}).get("total", 0) if closed else 0
    curve = equity(strat)
    years = (curve.index[-1] - curve.index[0]).days / 365.25
    return pd.Series(
        {
            "final value": strat.broker.getvalue(),
            "CAGR %": 100 * ((curve.iloc[-1] / strat.broker.startingcash) ** (1 / years) - 1),
            "Sharpe": a.sharpe.get_analysis()["sharperatio"],
            "max drawdown %": a.drawdown.get_analysis().max.drawdown,
            "longest drawdown (days)": a.timedd.get_analysis()["maxdrawdownperiod"],
            "trades": closed,
            "win rate %": 100 * won / closed if closed else float("nan"),
            "SQN": a.sqn.get_analysis().sqn,
            "VWR": a.vwr.get_analysis()["vwr"],
            "avg gross leverage": pd.Series(a.leverage.get_analysis()).mean(),
        },
        name=name,
    )


def backtest(strategy, datas, cash=CASH, commission=0.001, **params):
    cerebro = bt.Cerebro()
    for d in datas:
        cerebro.adddata(d)
    cerebro.addstrategy(strategy, **params)
    cerebro.broker.setcash(cash)
    cerebro.broker.setcommission(commission=commission)
    add_analyzers(cerebro)
    return cerebro, cerebro.run()[0]


class BuyAndHold(bt.Strategy):
    params = dict(target=0.99)

    def nextstart(self):
        weight = self.p.target / len(self.datas)
        for d in self.datas:
            self.order_target_percent(d, target=weight)


results = {}
_, s = backtest(BuyAndHold, [feed(A)])
results[f"{A} buy and hold"] = summary(s, f"{A} buy and hold")
cerebro_sma, sma_strat = backtest(SmaCross, [feed(A)])
results[f"{A} SMA 50/200"] = summary(sma_strat, f"{A} SMA 50/200")
pd.DataFrame(results).T

The raw output of any analyzer is a (nested) dictionary:

In [ ]:
tr = sma_strat.analyzers.trades.get_analysis()
print("trades:", {k: tr[k] for k in ("total", "streak")})
print("won:", dict(tr.won.pnl), "\nlost:", dict(tr.lost.pnl))
print("drawdown:", dict(sma_strat.analyzers.drawdown.get_analysis().max))
print("per year:", {y: round(r, 4) for y, r in sma_strat.analyzers.annual.get_analysis().items()})
print("period stats:", dict(sma_strat.analyzers.periods.get_analysis()))
txn = sma_strat.analyzers.transactions.get_analysis()
print("first transactions:", list(txn.items())[:2])

In [ ]:
monthly = pd.Series(sma_strat.analyzers.monthly.get_analysis())
monthly.index = pd.to_datetime(monthly.index)
grid = monthly.groupby([monthly.index.year, monthly.index.month]).first().unstack() * 100
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), width_ratios=[2, 1])
im = ax1.imshow(grid.values, cmap="RdYlGn", vmin=-15, vmax=15, aspect="auto")
ax1.set_xticks(range(12), ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax1.set_yticks(range(len(grid)), grid.index)
for (i, j), v in np.ndenumerate(grid.values):
    if not np.isnan(v):
        ax1.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7)
ax1.set_title(f"{A} SMA 50/200: monthly returns, %")
ax1.grid(False)
annual = pd.Series(sma_strat.analyzers.annual.get_analysis()) * 100
ax2.bar(annual.index.astype(str), annual.values, color=["green" if v > 0 else "red" for v in annual])
ax2.set_title("annual returns, %")
ax2.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

### 7.2 Observers

An **observer** is a line computed alongside the strategy, bar by bar, and
drawn by `cerebro.plot()`. Cerebro adds `Broker` (cash and value),
`BuySell` and `Trades` by default (`stdstats=True`). Others are added by
hand: `DrawDown`, `TimeReturn`, and `Benchmark`, which plots the
strategy's returns against another feed's.

In [ ]:
cerebro = bt.Cerebro(stdstats=False)
data = feed(A)
cerebro.adddata(data)
cerebro.addstrategy(SmaCross)
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)
cerebro.addobserver(bt.observers.Value)
cerebro.addobserver(bt.observers.BuySell, barplot=True)
cerebro.addobserver(bt.observers.DrawDown)
cerebro.addobserver(bt.observers.Benchmark, data=data, timeframe=bt.TimeFrame.Years)
st = cerebro.run()[0]

dd = [o for o in st.getobservers() if isinstance(o, bt.observers.DrawDown)][0]
print(f"deepest drawdown seen by the observer: {max(dd.lines.maxdrawdown.array):.2f}%")
show(cerebro, volume=False)

### 7.3 Plotting your own way

The analyzers hand back plain data, so any chart is a few lines of pandas
and matplotlib away: here the equity curves and the drawdowns of the two
runs above.

In [ ]:
curves = {name: equity(s) for name, s in [(f"{A} buy and hold", backtest(BuyAndHold, [feed(A)])[1]), (f"{A} SMA 50/200", sma_strat)]}
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, height_ratios=[2, 1])
for name, c in curves.items():
    ax1.plot(c.index, c, label=name)
    ax2.fill_between(c.index, 100 * (c / c.cummax() - 1), 0, alpha=0.4, label=name)
ax1.set_yscale("log")
ax1.set_title("account value")
ax1.legend()
ax2.set_title("drawdown, %")
plt.tight_layout()
plt.show()

## 8. Several stocks

With more than one feed, `self.datas` holds them all in the order they were
added (`self.data0`, `self.data1`, ... are shortcuts), `getdatabyname()`
finds one by name, and every order method takes the feed as its first
argument. `next()` runs once per bar of the *combined* timeline, after
every feed has delivered what it has for that moment.

### 8.1 Equal weight, rebalanced monthly on a timer

`add_timer()` calls `notify_timer()` on a schedule instead of on every bar:
here on the first trading day of each month (`monthcarry=True` moves the
call to the next trading day when the 1st is a holiday or weekend).

In [ ]:
class EqualWeight(bt.Strategy):
    params = dict(weights=None)

    def __init__(self):
        self.add_timer(when=bt.Timer.SESSION_START, monthdays=[1], monthcarry=True)
        self.rebalances = 0

    def notify_timer(self, timer, when, *args, **kwargs):
        weights = self.p.weights or {d._name: 0.99 / len(self.datas) for d in self.datas}
        for name, w in weights.items():
            self.order_target_percent(self.getdatabyname(name), target=w)
        self.rebalances += 1


cerebro_ew, ew = backtest(EqualWeight, [feed(A), feed(B)])
results["50/50 monthly rebalance"] = summary(ew, "50/50 monthly rebalance")
_, bh2 = backtest(BuyAndHold, [feed(A), feed(B)])
results["50/50 buy and hold"] = summary(bh2, "50/50 buy and hold")
print(f"{ew.rebalances} rebalances")
pd.DataFrame(results).T.tail(2)

### 8.2 Momentum rotation

Once a month, hold whichever of the two stocks rose more over the last six
months - and nothing if both fell. `RateOfChange` gives the six-month
return as an indicator line per feed.

In [ ]:
class Rotation(bt.Strategy):
    params = dict(lookback=126)

    def __init__(self):
        self.mom = {d: bt.ind.RateOfChange(d.close, period=self.p.lookback) for d in self.datas}
        self.add_timer(when=bt.Timer.SESSION_START, monthdays=[1], monthcarry=True)
        self.held = []

    def notify_timer(self, timer, when, *args, **kwargs):
        if len(self) <= self.p.lookback:
            return
        best = max(self.datas, key=lambda d: self.mom[d][0])
        pick = best if self.mom[best][0] > 0 else None
        for d in self.datas:  # sell first, so the cash is there to buy with
            if d is not pick:
                self.order_target_percent(d, target=0.0)
        if pick is not None:
            self.order_target_percent(pick, target=0.98)
        self.held.append((self.data.datetime.date(0), pick._name if pick else "cash"))


_, rot = backtest(Rotation, [feed(A), feed(B)])
results["momentum rotation"] = summary(rot, "momentum rotation")
held = pd.DataFrame(rot.held, columns=["month", "holding"])
print("months held:", held.holding.value_counts().to_dict())
pd.DataFrame(results).T.tail(1)

### 8.3 A market-neutral pair

Two large technology stocks tend to move together. The pair trade bets on
the *ratio* of their prices returning to its recent average: when the
z-score of the ratio is above 2, sell the expensive one and buy the cheap
one in equal money amounts; close when it is back within 0.5. The book is
long one stock and short the other, so the market's direction largely
cancels out. This reuses the `ZScore` indicator from section 2.4 on a line
computed from two feeds.

In [ ]:
class Pair(bt.Strategy):
    params = dict(period=60, entry=2.0, exit=0.5, gross=0.8)

    def __init__(self):
        self.a, self.b = self.datas
        self.z = ZScore(self.a.close / self.b.close, period=self.p.period)
        self.state = 0
        self.log = []

    def next(self):
        z = self.z[0]
        each = self.p.gross / 2 * self.broker.getvalue()
        if self.state == 0 and abs(z) > self.p.entry:
            self.state = -1 if z > 0 else 1  # short the ratio when it is high
            self.order_target_value(self.a, target=self.state * each)
            self.order_target_value(self.b, target=-self.state * each)
            self.log.append((self.data.datetime.date(0), "open", round(z, 2)))
        elif self.state and abs(z) < self.p.exit:
            self.order_target_size(self.a, target=0)
            self.order_target_size(self.b, target=0)
            self.state = 0
            self.log.append((self.data.datetime.date(0), "close", round(z, 2)))


cerebro_pair, pair = backtest(Pair, [feed(A), feed(B)])
results["pair trade"] = summary(pair, "pair trade")
print(f"{len(pair.log)} entries and exits, e.g.", pair.log[:4])
pd.DataFrame(results).T.tail(1)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, s in [("50/50 monthly rebalance", ew), ("momentum rotation", rot), ("pair trade", pair), ("50/50 buy and hold", bh2)]:
    c = equity(s)
    ax.plot(c.index, c, label=name)
ax.set_yscale("log")
ax.set_title(f"several stocks: {A} and {B}")
ax.legend()
plt.show()

## 9. Signals, optimisation and writers

### 9.1 Signals

A **signal** is an indicator whose sign is the instruction: positive to be
long, negative to be short, zero to stand aside. `add_signal()` attaches
signals to cerebro, and a built-in `SignalStrategy` trades them, so no
`next()` needs writing at all. The kind decides how a signal is read:
`SIGNAL_LONG` only enters and exits longs, `SIGNAL_LONGSHORT` reverses
between long and short.

In [ ]:
class MacdSignal(bt.Indicator):
    lines = ("signal",)

    def __init__(self):
        macd = bt.ind.MACD(self.data)
        self.lines.signal = macd.macd - macd.signal


for kind, label in [(bt.SIGNAL_LONG, "MACD signal, long only"), (bt.SIGNAL_LONGSHORT, "MACD signal, long/short")]:
    cerebro = bt.Cerebro()
    cerebro.adddata(feed(B))
    cerebro.add_signal(kind, MacdSignal)
    cerebro.addsizer(bt.sizers.PercentSizerInt, percents=90)
    cerebro.broker.setcash(CASH)
    cerebro.broker.setcommission(commission=0.001)
    add_analyzers(cerebro)
    results[f"{B} {label}"] = summary(cerebro.run()[0], f"{B} {label}")
pd.DataFrame(results).T.tail(2)

### 9.2 Optimisation

`optstrategy()` takes ranges instead of values and runs every combination.
With `optreturn=True` (the default) each run comes back as a light object
holding its parameters and analyzers. `maxcpus=1` keeps it in one process,
which works everywhere; on Linux, `maxcpus=None` uses every core.

A heat map of the Sharpe ratio over the two windows shows how sensitive the
crossover is to its parameters - and the best cell of a grid like this is
an in-sample result, flattered by having been picked.

In [ ]:
cerebro = bt.Cerebro(maxcpus=1)
cerebro.adddata(feed(B))
cerebro.optstrategy(SmaCross, fast=[10, 20, 30, 50, 75], slow=[100, 150, 200, 250])
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)
cerebro.addanalyzer(bt.analyzers.SharpeRatio_A, _name="sharpe", timeframe=bt.TimeFrame.Days)
cerebro.addanalyzer(bt.analyzers.Returns, _name="returns", tann=252)
runs = cerebro.run()
grid = pd.DataFrame(
    [
        dict(fast=r[0].params.fast, slow=r[0].params.slow,
             sharpe=r[0].analyzers.sharpe.get_analysis()["sharperatio"],
             annual_return=100 * r[0].analyzers.returns.get_analysis()["rnorm"])
        for r in runs
    ]
)
table = grid.pivot(index="fast", columns="slow", values="sharpe")
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(table.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(table.columns)), table.columns)
ax.set_yticks(range(len(table.index)), table.index)
ax.set_xlabel("slow SMA")
ax.set_ylabel("fast SMA")
for (i, j), v in np.ndenumerate(table.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white")
ax.set_title(f"{B}: Sharpe ratio of the SMA crossover")
ax.grid(False)
plt.colorbar(im)
plt.show()
print(f"{len(runs)} runs; best:", grid.loc[grid.sharpe.idxmax()].to_dict())

### 9.3 Writing a run out

`addwriter()` streams every bar's values - data, indicators, observers -
as CSV, to a file or any stream, and appends the analyzers at the end.

In [ ]:
import io

out = io.StringIO()
cerebro = bt.Cerebro()
cerebro.adddata(feed(A, start=f"{year}-01-01", end=f"{year}-03-31"))
cerebro.addstrategy(SmaCross, fast=5, slow=20)
cerebro.addwriter(bt.WriterFile, out=out, csv=True, rounding=2)
cerebro.addanalyzer(bt.analyzers.SQN)
cerebro.run()
lines = out.getvalue().splitlines()
print(f"{len(lines)} lines; the header and the first bars:")
print("\n".join(line[:150] for line in lines[1:5]))

## 10. Option pricing

`backtrader.options.pricing` is plain Python with no dependencies:

- `bsm(right, S, K, T, r, sigma, q)`: Black-Scholes-Merton for a European
  option on a stock with a continuous dividend yield `q`, returning the
  price and the greeks;
- `black76(right, F, K, T, r, sigma)`: the same on a futures price;
- `binomial(...)`: a Cox-Ross-Rubinstein tree, American by default;
- `implied_vol(price, right, S, K, T, r, ...)`: the volatility at which a
  model reproduces a price.

`T` is in years, rates and volatilities are annual fractions. The greeks
come back per unit of the underlying, with vega per 1.00 of volatility and
theta per year; the `OptionGreeks` indicator and `PortfolioGreeks` observer
below rescale them to a vol point and a calendar day.

### 10.1 Pricing an option on today's stock

Volatility is the one input not on the screen. Here it is the stock's own
realised volatility: the annualised standard deviation of its daily log
returns over the last 21 and 63 trading days.

In [ ]:
from backtrader.options import pricing

rets = np.log(prices[A]["close"]).diff()
spot = float(prices[A]["close"].iloc[-1])
vol21 = float(rets.iloc[-21:].std() * np.sqrt(252))
vol63 = float(rets.iloc[-63:].std() * np.sqrt(252))
r, q, T = 0.04, 0.005, 30 / 365
K = round(spot / 5) * 5  # the nearest $5 strike
print(f"{A} at {spot:.2f}; realised volatility {vol21:.1%} (21 days), {vol63:.1%} (63 days)")

call = pricing.bsm("C", spot, K, T, r, vol63, q)
put = pricing.bsm("P", spot, K, T, r, vol63, q)
print(f"30-day {K} call: {call.price:.2f} (delta {call.delta:+.3f}), put: {put.price:.2f} (delta {put.delta:+.3f})")
parity = call.price - put.price - (spot * np.exp(-q * T) - K * np.exp(-r * T))
print(f"put-call parity, C - P - (S·e^-qT - K·e^-rT) = {parity:.2e}")
pd.DataFrame({"call": call._asdict(), "put": put._asdict()}).T

### 10.2 American options and the binomial tree

An American put can be exercised early, which is worth something when it
is deep in the money and rates are positive. The tree prices both styles;
the European tree converges on Black-Scholes as the steps increase.

In [ ]:
steps = [10, 25, 50, 100, 200, 400, 800]
deep = K * 1.2
euro = [pricing.binomial("P", spot, deep, 0.5, r, vol63, q, steps=n, american=False) for n in steps]
amer = [pricing.binomial("P", spot, deep, 0.5, r, vol63, q, steps=n) for n in steps]
exact = pricing.bsm("P", spot, deep, 0.5, r, vol63, q).price
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, euro, "o-", label="European, binomial")
ax.plot(steps, amer, "s-", label="American, binomial")
ax.axhline(exact, color="black", ls=":", label="Black-Scholes")
ax.set_xscale("log")
ax.set_xlabel("tree steps")
ax.set_title(f"6-month put struck at {deep:.0f} (20% in the money)")
ax.legend()
plt.show()
print(f"early-exercise premium: {amer[-1] - euro[-1]:.3f} on a {euro[-1]:.2f} option")

### 10.3 Implied volatility

Going backwards: from a price to the volatility that produces it. A price
below intrinsic value has no implied volatility, and says so with `NaN`.

In [ ]:
for vol in (0.15, 0.30, 0.60):
    p = pricing.bsm("C", spot, K, T, r, vol, q).price
    print(f"vol {vol:.0%} -> price {p:7.3f} -> implied {pricing.implied_vol(p, 'C', spot, K, T, r, q):.6f}")
p_am = pricing.binomial("P", spot, deep, 0.5, r, 0.35, q)
print(f"American put at 35% -> {p_am:.3f} -> implied (binomial) {pricing.implied_vol(p_am, 'P', spot, deep, 0.5, r, q, model='binomial'):.4f}")
print("a price below intrinsic:", pricing.implied_vol(1.0, "P", spot, deep, 0.5, r, q))

### 10.4 The greeks across strikes and time

Delta is the hedge ratio, gamma how fast it changes, vega the exposure to
volatility and theta the daily decay. All four depend on how far the
strike is and how long is left.

In [ ]:
strikes = np.linspace(spot * 0.7, spot * 1.3, 61)
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for days in (7, 30, 90, 365):
    g = [pricing.bsm("C", spot, k, days / 365, r, vol63, q) for k in strikes]
    axes[0, 0].plot(strikes, [x.delta for x in g], label=f"{days}d")
    axes[0, 1].plot(strikes, [x.gamma for x in g])
    axes[1, 0].plot(strikes, [x.vega / 100 for x in g])
    axes[1, 1].plot(strikes, [x.theta / 365 for x in g])
for ax, title in zip(axes.flat, ["delta", "gamma", "vega per vol point", "theta per day"]):
    ax.axvline(spot, color="grey", ls=":")
    ax.set_title(f"call {title}")
axes[0, 0].legend(title="to expiry")
plt.tight_layout()
plt.show()

### 10.5 Black-76

On a futures price the carry is already in the price, so Black-76 drops
the dividend and discounts only the payoff. With `F = S·e^((r-q)T)` it
agrees with Black-Scholes.

In [ ]:
F = spot * np.exp((r - q) * T)
b76 = pricing.black76("C", F, K, T, r, vol63)
print(f"Black-76 on F={F:.2f}: {b76.price:.6f}   Black-Scholes on S={spot:.2f}: {call.price:.6f}")

## 11. Options on a model feed

An option is a data feed of its own. `ModelOptionData` builds one from the
stock's feed: each bar of the stock becomes a bar of the option, priced by
Black-Scholes (or the binomial tree) at the stock's realised volatility
*up to the previous bar*, so no bar's prices know its own move. An
`OptionContract` describes the instrument - underlying, strike, expiry,
call or put, multiplier, American or European, cash or physical
settlement - and `cerebro.addoption()` adds the feed with its commission
scheme (`OptionCommInfo`: per contract, with an exercise/assignment fee).

At expiry the broker settles every open contract by itself: an
in-the-money **physical** option turns into a stock trade at the strike
(exercise for a long, assignment for a short), a **cash** option pays its
intrinsic value, and an out-of-the-money one expires worthless. Each leg
reaches `notify_order` with `order.info.settlement` saying which it was.

`add_monthly()` lists one contract per month and leg for a stock: listed
the day after the previous expiry, expiring on the third Friday, struck at
`moneyness` times the previous close. The strategies below act on a
contract's listing day.

In [ ]:
from backtrader.options import (
    CashCovered, GreeksAttribution, ModelOptionData, OptionCommInfo,
    OptionContract, OptionGreeks, PortfolioGreeks, StructureTrades,
)

RATE, DIVIDEND, SPREAD = 0.04, 0.005, 0.04


def third_friday(year, month):
    day = datetime.date(year, month, 15)
    return day + datetime.timedelta(days=(4 - day.weekday()) % 7)


def expiries(start, end):
    months = pd.period_range(start, end, freq="M")
    return [third_friday(m.year, m.month) for m in months]


def strike_step(price):
    return max([s for s in (0.5, 1.0, 2.5, 5.0, 10.0) if s <= price * 0.025] or [0.5])


def add_monthly(cerebro, ticker, under, legs, settlement="physical", start=None, end=None, spread=SPREAD):
    """One model-priced contract per expiry and (right, moneyness) leg."""
    start, end = start or OPTIONS_START, end or END
    step = strike_step(float(prices[ticker]["close"].loc[start:].iloc[0]))
    dates = expiries(start, end)
    last_day = pd.Timestamp(end).date()
    for prev, expiry in zip(dates, dates[1:]):
        if expiry > last_day:
            break
        for right, moneyness in legs:
            contract = OptionContract(ticker, 1.0, expiry, right, settlement=settlement)
            cerebro.addoption(
                ModelOptionData(
                    dataname=under, contract=contract,
                    listing=prev + datetime.timedelta(days=1),
                    moneyness=moneyness, strikestep=step,
                    rate=RATE, dividend=DIVIDEND, spread=spread,
                ),
                comminfo=OptionCommInfo(commission=0.65, exercise_fee=1.0),
            )


class OptionStrategy(bt.Strategy):
    """The common parts: find the stock and the options, act on listing days."""

    params = dict(ticker=None, invest=0.9)

    def __init__(self):
        self.stock = self.getdatabyname(self.p.ticker)
        self.options = [d for d in self.datas if hasattr(d, "contract")]
        self.events = []

    def lots(self, price=None):
        """How many 100-share lots `invest` of the account buys today (or
        secures at `price`, for a cash-secured put)."""
        price = price or self.stock.close[0]
        return int(self.p.invest * self.broker.getvalue() / (100 * price))

    def prenext(self):
        self.next()

    def listed_today(self, right=None, moneyness=None):
        today = self.stock.datetime.date(0)
        return [
            o for o in self.options
            if len(o) == 1 and o.datetime.date(0) == today
            and (right is None or o.contract.right == right)
            and (moneyness is None or o.p.moneyness == moneyness)
        ]

    def shares(self):
        return self.getposition(self.stock).size

    def notify_order(self, order):
        if order.status == order.Completed:
            self.events.append(dict(
                date=order.data.datetime.date(0), instrument=order.data._name,
                what=order.info.get("settlement") or ("buy" if order.isbuy() else "sell"),
                size=order.executed.size, price=order.executed.price,
            ))


def option_backtest(strategy, ticker, legs, settlement="physical", cash=CASH, margin=False, quotefill=None, **params):
    cerebro = bt.Cerebro()
    under = feed(ticker, start=OPTIONS_START)
    cerebro.adddata(under, name=ticker)
    add_monthly(cerebro, ticker, under, legs, settlement)
    cerebro.addstrategy(strategy, ticker=ticker, **params)
    cerebro.broker.setcash(cash)
    cerebro.broker.setcommission(commission=0.001, name=ticker)
    if margin:
        cerebro.broker.set_margin_model(CashCovered())
    if quotefill is not None:
        cerebro.broker.set_quotefill(quotefill)
    add_analyzers(cerebro)
    cerebro.addanalyzer(StructureTrades, _name="structures")
    return cerebro, cerebro.run()[0]


option_results = {}
_, s = backtest(BuyAndHold, [feed(A, start=OPTIONS_START)])
option_results[f"{A} buy and hold"] = summary(s, f"{A} buy and hold")
_, s = backtest(BuyAndHold, [feed(B, start=OPTIONS_START)])
option_results[f"{B} buy and hold"] = summary(s, f"{B} buy and hold")
print(f"{len(expiries(OPTIONS_START, END)) - 1} monthly expiries from {OPTIONS_START}")

### 11.1 Covered calls

Own the stock and sell a call 5% above the price every month. The premium
is income; the cost is the upside above the strike, which is given away
when the stock is called. After an assignment the strategy buys the stock
back (a "buy-write").

In [ ]:
class CoveredCall(OptionStrategy):
    def next(self):
        if not self.shares() and not self.open_short_calls():
            self.buy(self.stock, size=100 * self.lots())  # (re)buy the stock
        for call in self.listed_today("C"):
            if self.shares() >= 100:
                self.sell(call, size=self.shares() // 100)

    def open_short_calls(self):
        return any(self.getposition(o).size < 0 for o in self.options)


cc_cerebro, cc = option_backtest(CoveredCall, A, [("C", 1.05)])
option_results[f"{A} covered call (5% OTM)"] = summary(cc, f"{A} covered call (5% OTM)")
ev = pd.DataFrame(cc.events)
print(ev.what.value_counts().to_dict())
ev[ev.what != "buy"].head(8)

### 11.2 Protective puts

Own the stock and buy a put 5% below the price each month: insurance that
caps the loss of any month near 5%, paid for with the premium every month.

In [ ]:
class ProtectivePut(OptionStrategy):
    params = dict(invest=0.85)  # leave room for the premiums

    def next(self):
        if not self.shares():
            self.buy(self.stock, size=100 * self.lots())
        for put in self.listed_today("P"):
            if self.shares() >= 100:
                self.buy(put, size=self.shares() // 100)


_, pp = option_backtest(ProtectivePut, B, [("P", 0.95)], settlement="cash")
option_results[f"{B} protective put (5% OTM)"] = summary(pp, f"{B} protective put (5% OTM)")
pd.DataFrame(option_results).T

### 11.3 The wheel: cash-secured puts, then covered calls

Sell a put 5% below the price, with the cash to buy the stock set aside.
If it is assigned, the strategy owns the stock and sells calls 5% above
until the stock is called away; then it starts over with puts.

The `CashCovered` margin model enforces "secured": a short put must have
its strike's worth of cash behind it, a short call the stock (or a long
call). An order that would break that is refused with status `Margin`.

In [ ]:
class Wheel(OptionStrategy):
    params = dict(invest=0.95)

    def next(self):
        for o in self.listed_today():
            if o.contract.isput and self.shares() == 0:
                self.sell(o, size=self.lots(price=o.contract.strike))
            elif o.contract.iscall and self.shares() >= 100:
                self.sell(o, size=self.shares() // 100)


_, wheel = option_backtest(Wheel, A, [("P", 0.95), ("C", 1.05)], margin=True)
option_results[f"{A} wheel (5% OTM)"] = summary(wheel, f"{A} wheel (5% OTM)")
ev = pd.DataFrame(wheel.events)
print("events:", ev.what.value_counts().to_dict())
refused = [o for o in wheel.broker.orders if o.status == o.Margin]
print(f"orders refused by the margin model: {len(refused)}")

### 11.4 A bull call spread, as one combo order

Buy the at-the-money call and sell the call 5% higher: a cheaper bet on a
rise, with the gain capped at the gap between the strikes. `combo()` sends
both legs as one order that fills completely or not at all, and the
margin model checks them together, so the bought call covers the sold one.
These contracts settle in cash.

In [ ]:
class BullCallSpread(OptionStrategy):
    params = dict(budget=0.10)

    def next(self):
        atm = self.listed_today("C", 1.0)
        otm = self.listed_today("C", 1.05)
        if atm and otm:
            debit = (atm[0].close[0] - otm[0].close[0]) * 100
            n = max(1, int(self.p.budget * self.broker.getvalue() / debit)) if debit > 0 else 1
            self.combo([(atm[0], n), (otm[0], -n)])


_, spread = option_backtest(BullCallSpread, B, [("C", 1.0), ("C", 1.05)], settlement="cash", margin=True)
option_results[f"{B} bull call spread (10% a month)"] = summary(spread, f"{B} bull call spread (10% a month)")
st = pd.DataFrame(spread.analyzers.structures.get_analysis()).T
print(f"{len(st)} spreads, {int((st.pnl > 0).sum())} profitable; best {st.pnl.max():,.0f}, worst {st.pnl.min():,.0f}")
st[["opened", "closed", "pnl", "worst"]].head()

### 11.5 A collar

Own the stock, buy a put 5% below and sell a call 5% above: the call's
premium pays for most of the put's. The stock's monthly result is boxed
into roughly ±5%.

In [ ]:
class Collar(OptionStrategy):
    def next(self):
        if not self.shares() and not any(self.getposition(o).size for o in self.options):
            self.buy(self.stock, size=100 * self.lots())
        put, call = self.listed_today("P", 0.95), self.listed_today("C", 1.05)
        if put and call and self.shares() >= 100:
            n = self.shares() // 100
            self.combo([(put[0], n), (call[0], -n)])


_, collar = option_backtest(Collar, A, [("P", 0.95), ("C", 1.05)], margin=True)
option_results[f"{A} collar (±5%)"] = summary(collar, f"{A} collar (±5%)")
pd.DataFrame(option_results).T

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, s in [(f"{A} covered call", cc), (f"{B} protective put", pp), (f"{A} wheel", wheel),
                (f"{B} bull call spread", spread), (f"{A} collar", collar)]:
    c = equity(s)
    ax.plot(c.index, c, label=name)
for t in (A, B):
    px = prices[t]["close"].loc[OPTIONS_START:]
    ax.plot(px.index, CASH * px / px.iloc[0], ls=":", label=f"{t} stock")
ax.set_title("option strategies on model-priced contracts")
ax.legend(ncol=2)
plt.show()

### 11.6 Watching one contract: the `OptionGreeks` indicator

`OptionGreeks(option, stock)` inverts the option's price into an implied
volatility every bar and derives the greeks from it, in broker-screen
units: vega per vol point, theta per calendar day. On a model feed the
implied volatility is, by construction, the realised volatility the model
was given - a useful check that the inversion works.

In [ ]:
class Watch(bt.Strategy):
    def __init__(self):
        self.g = OptionGreeks(self.datas[1], self.datas[0], rate=RATE, dividend=DIVIDEND)
        self.rows = []

    def next(self):
        o = self.datas[1]
        if len(o) and o.datetime.date(0) == self.data0.datetime.date(0):  # listed, not yet expired
            self.rows.append(dict(date=o.datetime.date(0), stock=self.data0.close[0], option=o.close[0],
                                  iv=self.g.iv[0], delta=self.g.delta[0], gamma=self.g.gamma[0],
                                  vega=self.g.vega[0], theta=self.g.theta[0]))


cerebro = bt.Cerebro(stdstats=False)
under = feed(A, start=OPTIONS_START)
cerebro.adddata(under, name=A)
exp = expiries(OPTIONS_START, END)[3]
contract = OptionContract(A, 1.0, exp, "C")
cerebro.addoption(ModelOptionData(dataname=under, contract=contract, listing=exp - datetime.timedelta(days=60),
                                  moneyness=1.0, strikestep=strike_step(spot), rate=RATE, dividend=DIVIDEND))
cerebro.addstrategy(Watch)
w = pd.DataFrame(cerebro.run()[0].rows).set_index("date")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(w.index, w.option, label="call price")
axes[0].twinx().plot(w.index, w.stock, color="grey", ls=":")
axes[0].set_title(f"{contract.underlying} {exp} call, strike {cerebro.datas[1].contract.strike:g}")
axes[1].plot(w.index, w.delta, label="delta")
axes[1].plot(w.index, w.iv, label="implied vol")
axes[1].legend()
axes[2].plot(w.index, w.theta, label="theta / day", color="red")
axes[2].legend()
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
w.tail(3)

## 12. An options book: condors, margin, quotes and greeks

A short **iron condor** sells a put spread below the price and a call
spread above it: it earns the credit if the stock stays inside the short
strikes, and loses at most the width of a wing minus that credit. Four
legs, so this section uses everything the options layer has for a book:

- **`combo()`**: the four legs go in as one order, filled completely or
  not at all;
- **`CashCovered`**: the margin model reserves, for each short option,
  what covers it - the stock, a long option of the same kind, or cash -
  and refuses orders it cannot cover. A naked short (no wings) is
  refused; with the wings, only the spread's width is reserved;
  `notify_margin()` hears about a margin call if the account falls short;
- **quote fills** (`set_quotefill`): market orders fill between the bid
  and the ask, at `mid ± fraction × half-spread` - 0 is mid, 1 crosses
  the spread;
- **`PortfolioGreeks`**: an observer that carries the whole book's delta,
  gamma, vega and theta in money terms, bar by bar;
- **`StructureTrades`**: profit and loss per combo, not per leg;
- **`GreeksAttribution`**: each day's change in the book split into
  delta, gamma, vega, theta and a residual.

In [ ]:
WINGS = {("P", 0.90): +1, ("P", 0.95): -1, ("C", 1.05): -1, ("C", 1.10): +1}


class IronCondor(OptionStrategy):
    params = dict(risk=0.10, try_naked=True)

    def __init__(self):
        super().__init__()
        self.refused, self.calls = [], []

    def next(self):
        legs = {(o.contract.right, o.p.moneyness): o for o in self.listed_today()}
        if len(legs) < 4:
            return
        if self.p.try_naked:  # once: show the margin model refusing it
            self.p.try_naked = False
            self.combo([(legs[("P", 0.95)], -1), (legs[("C", 1.05)], -1)])
        width = (legs[("C", 1.10)].contract.strike - legs[("C", 1.05)].contract.strike) * 100
        n = max(1, int(self.p.risk * self.broker.getvalue() / width))
        self.combo([(legs[key], side * n) for key, side in WINGS.items()])

    def notify_order(self, order):
        super().notify_order(order)
        if order.status == order.Margin:
            self.refused.append((self.stock.datetime.date(0), order.data._name, order.size))

    def notify_margin(self, call):
        self.calls.append((call.dt, call.shortfall))


def condor_run(quotefill, **params):
    cerebro = bt.Cerebro()
    under = feed(B, start=OPTIONS_START)
    cerebro.adddata(under, name=B)
    add_monthly(cerebro, B, under, list(WINGS), settlement="cash")
    cerebro.addstrategy(IronCondor, ticker=B, **params)
    cerebro.broker.setcash(CASH)
    cerebro.broker.set_margin_model(CashCovered())
    cerebro.broker.set_quotefill(quotefill)
    cerebro.addobserver(PortfolioGreeks, rate=RATE, dividend=DIVIDEND)
    add_analyzers(cerebro)
    cerebro.addanalyzer(StructureTrades, _name="structures")
    cerebro.addanalyzer(GreeksAttribution, _name="attribution", rate=RATE, dividend=DIVIDEND)
    return cerebro, cerebro.run()[0]


condor_cerebro, condor = condor_run(0.5)
option_results[f"{B} iron condors"] = summary(condor, f"{B} iron condors")
print("refused by the margin model:", condor.refused[:2])
print("margin calls:", condor.calls or "none")
structures = pd.DataFrame(condor.analyzers.structures.get_analysis()).T
print(f"{len(structures)} condors: {int((structures.pnl > 0).sum())} won, "
      f"average {structures.pnl.mean():+,.0f}, worst {structures.pnl.min():+,.0f}")
structures[["opened", "closed", "pnl", "worst"]].tail()

The book's greeks through the run, from the `PortfolioGreeks` observer:
a short condor is short gamma and vega and long theta - it is paid to wait,
and hurt by large moves.

In [ ]:
pg = [o for o in condor.getobservers() if isinstance(o, PortfolioGreeks)][0]
days = pd.to_datetime([bt.num2date(x) for x in condor.datetime.array])
book = pd.DataFrame({k: getattr(pg.lines, k).array for k in ("delta", "gamma", "vega", "theta")}, index=days)
fig, axes = plt.subplots(2, 2, figsize=(14, 6), sharex=True)
labels = {"delta": "delta: stock-equivalent exposure, in money", "gamma": "gamma: change in that exposure for a 1% move",
          "vega": "vega: money per vol point", "theta": "theta: money per day"}
for ax, (k, label) in zip(axes.flat, labels.items()):
    ax.plot(book.index, book[k])
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(label)
plt.tight_layout()
plt.show()

Where the money came from, according to `GreeksAttribution`, and what the
spread costs: the same condors filled at mid, halfway across and all the
way across the quote.

In [ ]:
split = condor.analyzers.attribution.get_analysis()
parts = pd.Series({p: split[p] for p in GreeksAttribution.PARTS})
fills = {f: condor_run(f, try_naked=False)[0].broker.getvalue() for f in (0.0, 0.5, 1.0)}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.bar(parts.index, parts.values, color=["green" if v > 0 else "red" for v in parts])
ax1.set_title(f"{B} condors: profit by source")
ax2.bar([f"fill {f:g}" for f in fills], [v - CASH for v in fills.values()])
ax2.set_title("profit at different quote fills")
plt.tight_layout()
plt.show()
print("sum of the parts:", round(parts.sum(), 2), " account gain:", round(condor.broker.getvalue() - CASH, 2))

## 13. Whole option chains

A feed per contract is fine for a handful of contracts. A real strategy
looks at the whole chain - every strike and expiry - and picks from it.
`OptionChain` holds a chain per stock without making each contract a feed:
it loads one day's chain at a time from a **source** and hands out a
`ContractView` only for the contracts a strategy asks for. A view behaves
like a data feed everywhere that matters: orders, positions, margin,
settlement and the analyzers.

Sources: `PandasChainSource` (a DataFrame of real quotes),
`PartitionedChainSource` (one year or month in memory at a time),
`CSVChainSource`, and `ModelChainSource`, used here, which models a
strike grid on monthly expiries at the stock's realised volatility.

### 13.1 Looking at a chain

In [ ]:
from backtrader.options import ModelChainSource, OptionChain


def add_chain(cerebro, ticker, data, settlement="physical"):
    step = strike_step(float(prices[ticker]["close"].loc[OPTIONS_START:].iloc[0]))
    source = ModelChainSource(strikestep=step, width=0.25, months=3, rate=RATE, dividend=DIVIDEND, spread=SPREAD)
    chain = OptionChain(data, source, settlement=settlement, rate=RATE, dividend=DIVIDEND)
    cerebro.addchain(chain, comminfo=OptionCommInfo(commission=0.65, exercise_fee=1.0))


class LookAtChain(bt.Strategy):
    def next(self):
        if len(self) != 60:
            return
        chain = self.getchain(A)
        e = chain.expiries()
        print(f"{self.data.datetime.date(0)}: {A} at {self.data.close[0]:.2f}, expiries {e}")
        rows = []
        for k in chain.strikes(e[0], "C"):
            call, put = chain.contract(e[0], k, "C"), chain.contract(e[0], k, "P")
            rows.append(dict(strike=k, call_bid=call.bid[0], call_ask=call.ask[0],
                             put_bid=put.bid[0], put_ask=put.ask[0]))
        self.table = pd.DataFrame(rows).set_index("strike")
        for d in (0.5, 0.3, 0.1):
            v = self.getchain(A).select("C", dte=(20, 60), delta=d)
            print(f"  {d:.0%}-delta call: {v._name}, mid {v.mid():.2f}")
        v = chain.select("P", dte=(20, 60), moneyness=0.95)
        print(f"  5% out-of-the-money put: {v._name}, mid {v.mid():.2f}")


cerebro = bt.Cerebro(stdstats=False)
data = feed(A, start=OPTIONS_START)
cerebro.adddata(data, name=A)
add_chain(cerebro, A, data)
cerebro.addstrategy(LookAtChain)
look = cerebro.run()[0]
look.table.iloc[::2]

### 13.2 Selling 30-delta puts on both stocks

Every month, on each stock, sell the put closest to 30 delta with 20 to 50
days left, cash-secured. Assignment delivers the stock, which is then held
- so the book drifts into stock after falls.

In [ ]:
class PutsByDelta(bt.Strategy):
    params = dict(delta=-0.30, share=0.45)

    def __init__(self):
        self.month = None
        self.assigned = []

    def next(self):
        day = self.data0.datetime.date(0)
        if day.month == self.month:
            return
        self.month = day.month
        for d in self.datas:
            if self.getposition(d).size:
                self.close(d)  # sell stock left by an assignment
                continue
            put = self.getchain(d._name).select("P", dte=(20, 50), delta=self.p.delta)
            if put is not None:
                n = int(self.p.share * self.broker.getvalue() / (100 * put.contract.strike))
                if n:
                    self.combo([(put, -n)])

    def notify_order(self, order):
        if order.status == order.Completed and order.info.get("settlement") == "delivery":
            self.assigned.append((order.data._name, order.data.datetime.date(0)))


cerebro = bt.Cerebro()
for t in TICKERS:
    data = feed(t, start=OPTIONS_START)
    cerebro.adddata(data, name=t)
    add_chain(cerebro, t, data)
cerebro.addstrategy(PutsByDelta)
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)
cerebro.broker.set_margin_model(CashCovered())
cerebro.broker.set_quotefill(0.5)
add_analyzers(cerebro)
cerebro.addanalyzer(StructureTrades, _name="structures")
puts = cerebro.run()[0]
option_results["30-delta puts on both"] = summary(puts, "30-delta puts on both")
res = pd.DataFrame(puts.analyzers.structures.get_analysis()).T
print(f"{len(res)} puts sold, {int((res.pnl > 0).sum())} profitable, {len(puts.assigned)} assigned")

### 13.3 Stocks with an options overlay

The 50/50 portfolio of section 8.1, with a covered-call overlay: each
month, after rebalancing, sell 25-delta calls on the shares held. The
overlay trades upside for income; the margin model checks every call is
covered by stock.

In [ ]:
class OverlayCalls(EqualWeight):
    params = dict(delta=0.25)

    def notify_timer(self, timer, when, *args, **kwargs):
        super().notify_timer(timer, when, *args, **kwargs)
        self.pending_calls = True

    def next(self):
        if not getattr(self, "pending_calls", False):
            return
        self.pending_calls = False
        for d in self.datas:
            lots = int(self.getposition(d).size // 100)
            if lots and not any(self.getposition(o).size for o in self.broker.positions if getattr(o, "contract", None) and o.contract.underlying == d._name):
                call = self.getchain(d._name).select("C", dte=(20, 50), delta=self.p.delta)
                if call is not None:
                    self.sell(call, size=lots)


cerebro = bt.Cerebro()
for t in TICKERS:
    data = feed(t, start=OPTIONS_START)
    cerebro.adddata(data, name=t)
    add_chain(cerebro, t, data)
cerebro.addstrategy(OverlayCalls)
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)
cerebro.broker.set_margin_model(CashCovered())
cerebro.broker.set_quotefill(0.5)
add_analyzers(cerebro)
overlay = cerebro.run()[0]
option_results["50/50 with call overlay"] = summary(overlay, "50/50 with call overlay")
_, ew_opt = backtest(EqualWeight, [feed(A, start=OPTIONS_START), feed(B, start=OPTIONS_START)])
option_results["50/50 monthly rebalance"] = summary(ew_opt, "50/50 monthly rebalance")
pd.DataFrame(option_results).T.tail(3)

### 13.4 Delta hedging a short straddle

Sell the at-the-money call and put each month and hedge the book's delta
with the stock - but only when it drifts more than a threshold away from
zero, since hedging every day costs commissions. What is left is a bet on
volatility: the strategy earns the options' time value and pays for the
moves it has to chase. The book's delta comes from each option's implied
volatility, the same way `PortfolioGreeks` computes it.

In [ ]:
class HedgedStraddle(bt.Strategy):
    params = dict(threshold=0.10, share=0.5)

    def __init__(self):
        self.month = None
        self.hedges = 0
        self.deltas = []

    def book_delta(self):
        """Shares-equivalent delta of the options held, plus the stock."""
        total = self.getposition(self.data).size
        S = self.data.close[0]
        for d, pos in self.broker.positions.items():
            c = getattr(d, "contract", None)
            if c is None or not pos.size or not len(d):
                continue
            T = c.years_to_expiry(self.data.datetime.date(0))
            iv = pricing.implied_vol(d.close[0], c.right, S, c.strike, T, RATE, DIVIDEND)
            if iv == iv:
                total += pos.size * c.multiplier * pricing.bsm(c.right, S, c.strike, T, RATE, iv, DIVIDEND).delta
        return total

    def next(self):
        day = self.data.datetime.date(0)
        chain = self.getchain(self.data._name)
        if day.month != self.month:
            self.month = day.month
            call = chain.select("C", dte=(20, 50), moneyness=1.0)
            put = chain.select("P", dte=(20, 50), moneyness=1.0)
            if call is not None and put is not None:
                n = max(1, int(self.p.share * self.broker.getvalue() / (100 * self.data.close[0])))
                self.combo([(call, -n), (put, -n)])
                self.lots = n
        delta = self.book_delta()
        self.deltas.append((day, delta))
        lots = getattr(self, "lots", 1)
        if abs(delta) > self.p.threshold * 100 * lots:
            self.buy(self.data, size=-round(delta)) if delta < 0 else self.sell(self.data, size=round(delta))
            self.hedges += 1


cerebro = bt.Cerebro()
data = feed(A, start=OPTIONS_START)
cerebro.adddata(data, name=A)
add_chain(cerebro, A, data, settlement="cash")
cerebro.addstrategy(HedgedStraddle)
cerebro.broker.setcash(CASH)
cerebro.broker.setcommission(commission=0.001)
cerebro.broker.set_quotefill(0.5)
add_analyzers(cerebro)
hedged = cerebro.run()[0]
option_results[f"{A} hedged short straddle"] = summary(hedged, f"{A} hedged short straddle")
deltas = pd.Series(dict(hedged.deltas))
print(f"{hedged.hedges} hedge trades; before each day's hedge the book's delta was within "
      f"{deltas.abs().quantile(0.5):,.0f} shares on a median day and {deltas.abs().quantile(0.95):,.0f} on the 95th percentile")
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(pd.to_datetime(deltas.index), deltas.values)
ax.set_title(f"{A} short straddle: the book's delta in shares each day, before that day's hedge")
plt.show()

## 14. Summary

Every strategy in the notebook, side by side. The stock strategies ran
over the whole period; the option strategies over the last three years,
against buy-and-hold over the same three years. Reading the table:

- **CAGR** and **final value** are what the account did; **Sharpe** is
  return per unit of daily volatility, annualised; **max drawdown** the
  worst fall from a peak;
- **trades** counts closed round trips (a buy-and-hold has none);
- **avg gross leverage** is the notional of all positions, long and short,
  over the account value: 1.0 is fully invested, 0.8 for a pair 40% long
  and 40% short.

In [ ]:
stock_table = pd.DataFrame(results).T
option_table = pd.DataFrame(option_results).T
cols = ["final value", "CAGR %", "Sharpe", "max drawdown %", "trades", "win rate %", "avg gross leverage"]
print(f"STOCK STRATEGIES, {START} to {END}")
stock_table[cols].sort_values("Sharpe", ascending=False)

In [ ]:
print(f"OPTION STRATEGIES (model-priced options), {OPTIONS_START} to {END}")
option_table[cols].sort_values("Sharpe", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, table, title in [(axes[0], stock_table, "stock strategies"), (axes[1], option_table, "option strategies")]:
    t = table.dropna(subset=["Sharpe"])
    ax.scatter(t["max drawdown %"], t["CAGR %"], s=60)
    for name, row in t.iterrows():
        ax.annotate(name, (row["max drawdown %"], row["CAGR %"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel("max drawdown, %")
    ax.set_ylabel("CAGR, %")
    ax.set_title(title)
plt.tight_layout()
plt.show()

### What to take away

- **The engine is the same for every instrument.** A stock, a model-priced
  option and a contract picked from a chain all go through the same
  orders, broker, margin, settlement and analyzers, so a strategy that
  holds both needs nothing special.
- **Check the mechanics before the results.** The sections above checked
  the indicators against pandas, both execution paths against each other,
  fills against the bars they happened on, and the P&L attribution
  against the account. That is what makes the numbers worth reading.
- **Model-priced options flatter selling.** With no volatility risk
  premium, no skew and a fixed spread, the option sections show the shape
  of each strategy's risk, not what it would have earned. For that, feed
  real quotes through `PandasChainSource` or `OptionPandasData`.
- **Most of the stock strategies' differences are one stock's path.** Two
  tickers and a decade is one history; the optimisation heat map in 9.2
  shows how much a result moves with its parameters.